# ⚙️ Tool Calling

## What is Tool Calling?
**Tool calling** = the process by which an LLM decides to call a tool, and we execute it.

## The Tool Calling Loop
```
User: "What is 25 * 4?"
   ↓
LLM: "I'll use the calculator tool"
   → tool_calls: [{name: "calculate", args: {"expression": "25 * 4"}}]
   ↓
We execute calculate("25 * 4") → "100"
   ↓
LLM: "The answer is 100"
```

## Key Insight
The LLM does NOT directly execute code.
The LLM **requests** a tool call.
**Our code** executes it and returns the result.

## Function Calling vs Tool Calling
- **Function calling**: OpenAI's original term
- **Tool calling**: LangChain's standardized term (works across all providers)


In [ ]:
# ── Full Tool Calling Loop ─────────────────────────────────────────────
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

@tool
def search_database(query: str) -> str:
    """Search internal company database. Returns relevant records."""
    fake_db = {
        'sales': 'Q3 2024 sales: $2.5M, up 15% YoY',
        'employees': 'Total employees: 250, Engineering: 80',
        'products': 'Product catalog: 45 items, 3 categories',
    }
    for key, value in fake_db.items():
        if key in query.lower():
            return value
    return 'No matching records found'

tools = [search_database]
tool_map = {tool.name: tool for tool in tools}  # name → function mapping

llm = ChatOpenAI(model='gpt-4o-mini').bind_tools(tools)

# ── Step 1: Send user message ──
messages = [HumanMessage(content='How many employees does the company have?')]
ai_response = llm.invoke(messages)
print('Step 1 - AI Response:')
print(f'  Tool calls: {ai_response.tool_calls}')

# ── Step 2: Execute tool calls ──
messages.append(ai_response)
for tool_call in ai_response.tool_calls:
    tool_fn = tool_map[tool_call['name']]
    result = tool_fn.invoke(tool_call['args'])
    print(f'\nStep 2 - Tool execution:')
    print(f'  Tool: {tool_call["name"]}')
    print(f'  Args: {tool_call["args"]}')
    print(f'  Result: {result}')
    messages.append(ToolMessage(content=result, tool_call_id=tool_call['id']))

# ── Step 3: Final LLM response with tool results ──
final_response = llm.invoke(messages)
print(f'\nStep 3 - Final Answer:')
print(f'  {final_response.content}')

## 🎯 Interview Questions

1. **Does the LLM actually execute tool code? Explain.**
2. **What is a ToolMessage and when is it used?**
3. **Trace the tool calling loop step by step.**
4. **What is tool_call_id used for?**
5. **How do you handle tool execution errors?**

> Answer these before moving to the next module.

## 💪 Mini Exercise

**Exercise**: Based on what you learned in this module:

1. Re-run all code cells with your own inputs
2. Modify one code cell to add new functionality
3. Answer the interview questions above from memory

**Mini Project**: Build a small application that combines the concepts from this module.


## 📚 Summary

In this module you learned:
- The core concepts of **Tool Calling — LLMs Executing Functions**
- How to implement them with modern LangChain/LangGraph APIs
- Common mistakes and how to avoid them
- Interview-level understanding of why each component exists

**Next**: Continue to the next module to build on these foundations.

---
*Part of the LangChain & LangGraph Master Course*
